In [15]:
# ============================================================================
# COLAB SETUP — run this cell first
# ============================================================================
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # =========================================================================
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/ENERGIZE'  # <-- EDIT THIS
    # =========================================================================

    import os
    from pathlib import Path
    project_root = Path(DRIVE_PROJECT_PATH)
    if not project_root.exists():
        raise FileNotFoundError(f'Project folder not found: {project_root}')
    os.chdir(project_root)
    sys.path.insert(0, str(project_root))
    print(f'Project root: {project_root}')
else:
    import os
    from pathlib import Path
    project_root = Path(os.getcwd()).parent
    sys.path.insert(0, str(project_root))
    print(f'Running locally. Project root: {project_root}')

Running locally. Project root: c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-NILM-Compression-Pipeline


In [16]:
import numpy as np
import pandas as pd
from pathlib import Path

In [17]:
# ============================================================================
# CONFIGURATION — edit this cell
# ============================================================================
_MODEL       = 'tcn'     # 'cnn' | 'gru' | 'tcn'
_APPLIANCE   = 'boiler'  # 'boiler' | 'ac_1' | 'washing_machine' | 'fridge'
_PRUNING_PCT = 75        # pruning percentage, e.g. 75 means 75%
_DATASET     = 'plegma'  # 'plegma' | 'refit'
_PERIOD_SEC  = 10        # seconds per sample (PLEGMA=10, REFIT=8)

# ── Appliance post-processing params ──────────────────────────────────────
from src_pytorch.config import PLEGMA_PARAMS, REFIT_PARAMS

_ap         = {'plegma': PLEGMA_PARAMS, 'refit': REFIT_PARAMS}[_DATASET][_APPLIANCE]
_THRESHOLD  = _ap['threshold']
_MIN_ON     = _ap.get('min_on',  1)
_MIN_OFF    = _ap.get('min_off', 1)
_MAX_LENGTH = _ap.get('max_length', None)

# ── Predictions directory ─────────────────────────────────────────────────
_PRED_DIR = project_root / 'outputs' / f'{_MODEL}_{_APPLIANCE}' / 'predictions'

print(f'Model       : {_MODEL.upper()}')
print(f'Appliance   : {_APPLIANCE}')
print(f'Pruning     : {_PRUNING_PCT}%')
print(f'Pred dir    : {_PRED_DIR}')
print()
print('Post-processing params:')
print(f'  Threshold  = {_THRESHOLD} W')
print(f'  Min ON     = {_MIN_ON} samples  ({_MIN_ON * _PERIOD_SEC} s)')
print(f'  Min OFF    = {_MIN_OFF} samples  ({_MIN_OFF * _PERIOD_SEC} s)')
if _MAX_LENGTH:
    print(f'  Max Length = {_MAX_LENGTH} samples  ({_MAX_LENGTH * _PERIOD_SEC} s)')
else:
    print('  Max Length : disabled')

Model       : TCN
Appliance   : boiler
Pruning     : 75%
Pred dir    : c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-NILM-Compression-Pipeline\outputs\tcn_boiler\predictions

Post-processing params:
  Threshold  = 500 W
  Min ON     = 30 samples  (300 s)
  Min OFF    = 6 samples  (60 s)
  Max Length : disabled


In [22]:
# ============================================================================
# Load prediction CSVs
# Stages are tried in order; missing files are silently skipped.
# Both the new pipeline naming and the old notebook naming are supported.
# ============================================================================
_pct = _PRUNING_PCT

# (slug, display_label, colour)
_STAGE_DEFS = [
    ('baseline',
     'Baseline',
     '#2196F3'),
    ('baseline_quantized',
     'Baseline Quantized',
     '#00BCD4'),
    (f'unstructured_pruned_{_pct}pct_finetuned',
     f'Unstructured Pruned {_pct}% + Fine-tuned',
     '#FF9800'),
    (f'pruned_{_pct}pct_finetuned',
     f'Structured Pruned {_pct}% + Fine-tuned',
     '#4CAF50'),
    (f'pruned_{_pct}pct_int8',
     f'Structured Pruned {_pct}% + Fine-tuned + INT8',
     '#9C27B0'),
]

# Old-naming fallbacks: slug -> stem used in {model}_{appliance}_{stem}_preds.csv
_OLD_STEM = {
    'baseline':                                    'baseline',
    'baseline_quantized':                          'baseline_int8',   # <-- key was 'baseline_int8', must match the slug
    f'unstructured_pruned_{_pct}pct_finetuned':    f'unstructured_{_pct}pct_finetuned',
    f'pruned_{_pct}pct_finetuned':                 f'pruned_{_pct}pct_finetuned',
    f'pruned_{_pct}pct_int8':                      f'pruned_{_pct}pct_int8',
}


def _resolve_csv(slug: str) -> Path | None:
    # New convention: {slug}_predictions.csv
    p = _PRED_DIR / f'{slug}_predictions.csv'
    if p.exists():
        return p
    # Old convention: {model}_{appliance}_{stem}_preds.csv
    stem = _OLD_STEM.get(slug, slug)
    p = _PRED_DIR / f'{_MODEL}_{_APPLIANCE}_{stem}_preds.csv'
    if p.exists():
        return p
    return None


def _load_csv(path: Path) -> tuple:
    df = pd.read_csv(path)
    return (
        df['ground_truth_W'].values.astype('float32'),
        df['prediction_W'].values.astype('float32'),
    )


_loaded = {}   # slug -> {label, colour, gt, pred, path}
for slug, label, colour in _STAGE_DEFS:
    csv_path = _resolve_csv(slug)
    if csv_path is None:
        print(f'  [skip]  {label:<55}  (no file found)')
        continue
    gt, pred = _load_csv(csv_path)
    _loaded[slug] = dict(label=label, colour=colour, gt=gt, pred=pred, path=csv_path)
    print(f'  [ok]    {label:<55}  {len(gt):>10,} samples  ({csv_path.name})')

print()
if not _loaded:
    raise FileNotFoundError(
        f'No prediction files found in:\n  {_PRED_DIR}\n'
        'Run the pipeline steps first.'
    )

# Ground truth is shared across all stages (same test set)
_first_slug = next(iter(_loaded))
_GT  = _loaded[_first_slug]['gt']
_N   = len(_GT)
print(f'Ground truth    : {_N:,} samples  (~{_N * _PERIOD_SEC / 3600:.1f} h)')
print(f'Stages loaded   : {list(_loaded.keys())}')

  [ok]    Baseline                                                  1,626,600 samples  (tcn_boiler_baseline_preds.csv)
  [ok]    Baseline Quantized                                        1,626,600 samples  (tcn_boiler_baseline_int8_preds.csv)
  [ok]    Unstructured Pruned 75% + Fine-tuned                      1,626,600 samples  (tcn_boiler_unstructured_75pct_finetuned_preds.csv)
  [ok]    Structured Pruned 75% + Fine-tuned                        1,626,600 samples  (tcn_boiler_pruned_75pct_finetuned_preds.csv)
  [ok]    Structured Pruned 75% + Fine-tuned + INT8                 1,626,600 samples  (tcn_boiler_pruned_75pct_int8_preds.csv)

Ground truth    : 1,626,600 samples  (~4518.3 h)
Stages loaded   : ['baseline', 'baseline_quantized', 'unstructured_pruned_75pct_finetuned', 'pruned_75pct_finetuned', 'pruned_75pct_int8']


In [24]:
# ============================================================================
# Static PNG export (optional)
# Saves one figure per stage to outputs/{model}_{appliance}/figures/
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from src_pytorch.evaluator import compute_status

# Recompute status from current _loaded so this cell is self-contained
if '_GT_STATUS' not in dir():
    _GT_STATUS = compute_status(_GT, _THRESHOLD, _MIN_ON, _MIN_OFF, _MAX_LENGTH)

_pred_status = {
    slug: compute_status(info['pred'], _THRESHOLD, _MIN_ON, _MIN_OFF, _MAX_LENGTH)
    for slug, info in _loaded.items()
}

_EXPORT_START  = 0       # sample index to start
_EXPORT_LENGTH = 5_000   # number of samples
_EXPORT_DPI    = 150
# ... rest unchanged


In [26]:
# ============================================================================
# Static PNG export (optional)
# Saves one figure per stage to outputs/{model}_{appliance}/figures/
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

_EXPORT_START  = 0       # sample index to start
_EXPORT_LENGTH = 5_000   # number of samples
_EXPORT_DPI    = 150

_fig_dir = project_root / 'outputs' / f'{_MODEL}_{_APPLIANCE}' / 'figures'
_fig_dir.mkdir(parents=True, exist_ok=True)

_s = _EXPORT_START
_e = min(_s + _EXPORT_LENGTH, _N)
_x = np.arange(_s, _e)

_gt_slice    = _GT[_s:_e]
_gt_st_slice = _GT_STATUS[_s:_e]

for slug, info in _loaded.items():
    label  = info['label']
    colour = info['colour']
    pred   = info['pred'][_s:_e]
    st     = _pred_status[slug][_s:_e]

    fig, axes = plt.subplots(
        2, 1, figsize=(16, 6), sharex=True,
        gridspec_kw={'height_ratios': [2, 1]},
    )
    fig.suptitle(
        f'{_APPLIANCE.replace("_"," ").title()} — {_MODEL.upper()} | {label} vs Ground Truth\n'
        f'samples {_s:,}–{_e:,}',
        fontsize=11,
    )

    # Power
    ax = axes[0]
    ax.plot(_x, _gt_slice, color='#455A64', lw=1.0, label='Ground Truth')
    ax.plot(_x, pred,      color=colour,    lw=0.9, label=label)
    ax.axhline(_THRESHOLD, color='gray', lw=0.8, ls='--',
               label=f'Threshold ({_THRESHOLD} W)')
    ax.set_ylabel('Power (W)')
    ax.legend(loc='upper right', fontsize=8)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:.0f}'))

    # Status
    ax2 = axes[1]
    ax2.fill_between(_x, _gt_st_slice, alpha=0.40, color='#455A64', label='GT Status')
    ax2.fill_between(_x, st,           alpha=0.35, color=colour,    label='Pred Status')
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['OFF', 'ON'])
    ax2.set_xlabel('Sample index')
    ax2.legend(loc='upper right', fontsize=8)

    plt.tight_layout()

    fname = _fig_dir / f'gt_vs_{slug}.png'
    fig.savefig(fname, dpi=_EXPORT_DPI, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {fname.name}')

print('\nExport complete.')

  Saved: gt_vs_baseline.png
  Saved: gt_vs_baseline_quantized.png
  Saved: gt_vs_unstructured_pruned_75pct_finetuned.png
  Saved: gt_vs_pruned_75pct_finetuned.png
  Saved: gt_vs_pruned_75pct_int8.png

Export complete.


In [ ]:
# ============================================================================
# Interactive Explorer — Prediction vs Ground Truth, one figure per stage
# A single set of navigation sliders drives all figures simultaneously.
# ============================================================================

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError:
    raise ImportError('plotly not found — install with:  pip install plotly')

try:
    import ipywidgets as widgets
    from IPython.display import display as _ipy_display, clear_output
except ImportError:
    raise ImportError('ipywidgets not found — install with:  pip install ipywidgets')

from src_pytorch.evaluator import compute_status

# Recompute status from current _loaded so this cell is self-contained
_GT_STATUS = compute_status(_GT, _THRESHOLD, _MIN_ON, _MIN_OFF, _MAX_LENGTH)

_pred_status = {
    slug: compute_status(info['pred'], _THRESHOLD, _MIN_ON, _MIN_OFF, _MAX_LENGTH)
    for slug, info in _loaded.items()
}


# ── Shared navigation sliders ─────────────────────────────────────────────
_wl = widgets.Layout(width='100%')
_ws = {'description_width': '95px'}

_w_start = widgets.IntSlider(
    value=0, min=0, max=max(0, _N - 500), step=500,
    description='Start', continuous_update=False, layout=_wl, style=_ws,
)
_w_len = widgets.IntSlider(
    value=5_000, min=500, max=min(50_000, _N), step=500,
    description='Length', continuous_update=False, layout=_wl, style=_ws,
)


# ── Build a fresh go.Figure for a given stage and window ─────────────────
def _make_fig(slug: str, start: int, end: int) -> go.Figure:
    info   = _loaded[slug]
    colour = info['colour']
    label  = info['label']
    x      = list(range(start, end))

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=['Power (W)', 'ON / OFF Status'],
        vertical_spacing=0.28,
        row_heights=[0.65, 0.35],
    )

    # Row 1 — Power
    fig.add_trace(go.Scatter(
        x=x, y=_GT[start:end].tolist(), name='Ground Truth',
        line=dict(color='#455A64', width=1.4),
        hovertemplate='GT: %{y:.1f} W<extra></extra>',
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=x, y=info['pred'][start:end].tolist(), name='Prediction',
        line=dict(color=colour, width=1),
        hovertemplate='Pred: %{y:.1f} W<extra></extra>',
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[x[0], x[-1]], y=[_THRESHOLD, _THRESHOLD], name='Threshold',
        line=dict(color='gray', dash='dash', width=1),
        hoverinfo='skip', showlegend=False,
    ), row=1, col=1)

    # Row 2 — ON/OFF Status
    fig.add_trace(go.Scatter(
        x=x, y=_GT_STATUS[start:end].tolist(), name='GT Status',
        fill='tozeroy', fillcolor='rgba(69,90,100,0.30)',
        line=dict(color='#455A64', width=2.0),
        hovertemplate='GT: %{y}<extra></extra>',
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=x, y=_pred_status[slug][start:end].tolist(), name='Pred Status',
        fill='tozeroy',
        line=dict(color=colour, width=1.2),
        hovertemplate='Pred: %{y}<extra></extra>',
    ), row=2, col=1)

    fig.update_layout(
        height=560,
        margin=dict(t=120, b=40, l=60, r=20),
        title=dict(
            text=(
                f'<b>{_APPLIANCE.replace("_"," ").title()} \u2014 '
                f'{_MODEL.upper()} \u2014 {label} vs Ground Truth</b>'
                f'  |  samples {start:,}\u2013{end:,} / {_N:,}'
            ),
            y=0.98, yanchor='top',
        ),
        legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='right', x=1),
        hovermode='x unified',
    )
    fig.update_yaxes(title_text='Power (W)', row=1, col=1)
    fig.update_yaxes(
        title_text='Status', tickvals=[0, 1], ticktext=['OFF', 'ON'],
        range=[-0.15, 1.15], row=2, col=1,
    )
    fig.update_xaxes(title_text='Sample index', row=2, col=1)

    return fig


# ── One Output widget per stage ───────────────────────────────────────────
_out_widgets = {slug: widgets.Output() for slug in _loaded}


# ── Refresh — redraws every figure inside its Output widget ──────────────
def _refresh(_change=None):
    start = _w_start.value
    end   = min(start + _w_len.value, _N)

    for slug, out in _out_widgets.items():
        fig = _make_fig(slug, start, end)
        with out:
            clear_output(wait=True)
            fig.show()


for _w in [_w_start, _w_len]:
    _w.observe(_refresh, names='value')


# ── Layout: sidebar sliders + stacked Output widgets ─────────────────────
_sidebar = widgets.VBox(
    [
        widgets.HTML('<b style="font-size:13px">Navigation (shared across all figures)</b>'),
        _w_start,
        _w_len,
    ],
    layout=widgets.Layout(
        width='340px', min_width='340px',
        padding='10px', margin='0 10px 0 0',
        border='1px solid #ddd',
    ),
)

_figures_col = widgets.VBox(
    list(_out_widgets.values()),
    layout=widgets.Layout(flex='1 1 auto', min_width='0'),
)

_ipy_display(widgets.HBox([_sidebar, _figures_col]))
_refresh()

In [30]:
# ============================================================================
# Static PNG export (optional)
# Saves one figure per stage to outputs/{model}_{appliance}/figures/
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

_EXPORT_START  = 5500       # sample index to start
_EXPORT_LENGTH = 1000   # number of samples
_EXPORT_DPI    = 150

_fig_dir = project_root / 'outputs' / f'{_MODEL}_{_APPLIANCE}' / 'figures'
_fig_dir.mkdir(parents=True, exist_ok=True)

_s = _EXPORT_START
_e = min(_s + _EXPORT_LENGTH, _N)
_x = np.arange(_s, _e)

_gt_slice    = _GT[_s:_e]
_gt_st_slice = _GT_STATUS[_s:_e]

for slug, info in _loaded.items():
    label  = info['label']
    colour = info['colour']
    pred   = info['pred'][_s:_e]
    st     = _pred_status[slug][_s:_e]

    fig, axes = plt.subplots(
        2, 1, figsize=(16, 6), sharex=True,
        gridspec_kw={'height_ratios': [2, 1]},
    )
    fig.suptitle(
        f'{_APPLIANCE.replace("_"," ").title()} — {_MODEL.upper()} | {label} vs Ground Truth\n'
        f'samples {_s:,}–{_e:,}',
        fontsize=11,
    )

    # Power
    ax = axes[0]
    ax.plot(_x, _gt_slice, color='#455A64', lw=1.0, label='Ground Truth')
    ax.plot(_x, pred,      color=colour,    lw=0.9, label=label)
    ax.axhline(_THRESHOLD, color='gray', lw=0.8, ls='--',
               label=f'Threshold ({_THRESHOLD} W)')
    ax.set_ylabel('Power (W)')
    ax.legend(loc='upper right', fontsize=8)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:.0f}'))

    # Status
    ax2 = axes[1]
    ax2.fill_between(_x, _gt_st_slice, alpha=0.40, color='#455A64', label='GT Status')
    ax2.fill_between(_x, st,           alpha=0.35, color=colour,    label='Pred Status')
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['OFF', 'ON'])
    ax2.set_xlabel('Sample index')
    ax2.legend(loc='upper right', fontsize=8)

    plt.tight_layout()

    fname = _fig_dir / f'gt_vs_{slug}.png'
    fig.savefig(fname, dpi=_EXPORT_DPI, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {fname.name}')

print('\nExport complete.')

  Saved: gt_vs_baseline.png
  Saved: gt_vs_baseline_quantized.png
  Saved: gt_vs_unstructured_pruned_75pct_finetuned.png
  Saved: gt_vs_pruned_75pct_finetuned.png
  Saved: gt_vs_pruned_75pct_int8.png

Export complete.
